In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("world_bank_data_2025.csv")
df['Log_GDP_per_Capita'] = np.log(df['GDP per Capita (Current USD)'])

features = [
    'Interest Rate (Real, %)',
    'Government Expense (% of GDP)',
    'Tax Revenue (% of GDP)',
    'Gross National Income (USD)',
    'GDP per Capita (Current USD)',
    'Unemployment Rate (%)'
]

clean_df = df[features].dropna().copy()

clean_df['LogPerCapita'] = np.log(clean_df['GDP per Capita (Current USD)'])

X_features = [
    'Interest Rate (Real, %)',
    'Government Expense (% of GDP)',
    'Gross National Income (USD)',
    'Tax Revenue (% of GDP)',
    'Unemployment Rate (%)'
    ##'GDP per Capita (Current USD)' - didn't think about this is going to be quite redundant, I'm removing from features now
]

clean_df['Income_Tier'] = pd.qcut(clean_df['LogPerCapita'], q=3, labels=[0, 1, 2])

x = clean_df[X_features]
y = clean_df['Income_Tier']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print('Training features shape:', X_train.shape)
print('Testing features shape:', X_test.shape)

Training features shape: (877, 5)
Testing features shape: (220, 5)


# Pick an initial set of features for X and the target feature y.  Explain why you made these choices.
My target feature is Income_Tier (which is buckets for logs of GDP per Capita).

X features are Interest Rate, Government Expense, Tax Revenue, Inflation, and Unemployement Rate. 

I chose the target (y) feature to be buckets for log of GDP per Capita because GDP Per Capita is a continuous variable and performing log on GDP per Capita aids in consolidating the data while not losing it's value/meaning. I chose the x features because these were shown to be among the highest potential/correlation alongside Log GDP per Capita. They also give a good range of information about the state of a country's economy.

In [ ]:

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)

dt_pred = dt_clf.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

print(f"Decision Tree Classifier Accuracy: {dt_accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, dt_pred))

Decision Tree Classifier Accuracy: 0.7773
Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.80      0.76        66
           1       0.83      0.69      0.75        80
           2       0.78      0.85      0.81        74

    accuracy                           0.78       220
   macro avg       0.78      0.78      0.78       220
weighted avg       0.78      0.78      0.78       220



The decision tree classifier accuracy reading as 77.73% is really pretty good, given the vast amount of countries and their GDP tiers. It seems that this model is best (going off the f1 score) at predicting countries in the wealthiest tier with slightly less efficacy in the middle and lower GDP tiers. This is not surprising as generally the weathlier countries have consistent data associated with them in comparison to countries who's GDP is quite a bit lower.

In [5]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

svc_clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=42))

svc_clf.fit(X_train, y_train)

svc_pred = svc_clf.predict(X_test)
svc_accuracy = accuracy_score(y_test, svc_pred)

print(f"SVC Classifier Accuracy: {svc_accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, svc_pred))

SVC Classifier Accuracy: 0.6909

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.79      0.68        66
           1       0.67      0.60      0.63        80
           2       0.85      0.70      0.77        74

    accuracy                           0.69       220
   macro avg       0.71      0.70      0.69       220
weighted avg       0.71      0.69      0.69       220



# Look at the parameters you found and discuss what you have learned.
Interestingly enough, the decision tree classifier performed better than the SVM classifier did. This could be because the separation of the decision boundaries based on the levels of income are more hierarchal which tends to be captured more effectively by decision tree classifiers. The Support Vector Machine classifier does use a standard scaler which is helpful when working with numbers that operate on vastly different scales such as single digit percentages and features like Gross National Income which is quite large. The SVM predicted the data better than random guessing but did fall short in comparson to the Decision tree. If there was a greater number of quartiles for 'Income_Tier', the SVM may be more effective. 